# Uncertainty, Confidence Intervals, and Hypothesis Testing

## Learning Objectives

By the end of this notebook, you should be able to:
- explain why fitted regression coefficients are estimates rather than exact values
- calculate and interpret a confidence interval for a slope
- distinguish a confidence interval from a prediction interval
- state and test a null hypothesis for a linear relationship
- interpret a p-value

We will use the same temperature with elevation example as the linear regression and residuals notebook, and ask the question:

How certain are we about the model fit?

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

## Temperature Change with Elevation

In [ ]:
rng = np.random.default_rng(10)

elevation = np.linspace(0, 2500, 50)
temperature = 24 - 0.0065 * elevation + rng.normal(0, 1.5, len(elevation))

df = pd.DataFrame({
    "elevation_m": elevation,
    "temperature_C": temperature
})

df.head()

## Fit the Linear Model

As before, the model fit is

$$
\hat{y} = b_0 + b_1x
$$

$b_0$ is the intercept and $b_1$ is the slope

In [ ]:
slope, intercept = np.polyfit(df["elevation_m"], df["temperature_C"], 1)
df["predicted_C"] = slope * df["elevation_m"] + intercept
df["residual_C"] = df["temperature_C"] - df["predicted_C"]

print(f"Slope: {slope:.5f} °C per meter")
print(f"Intercept: {intercept:.2f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(df["elevation_m"], df["temperature_C"], label="Observed")
ax.plot(df["elevation_m"], df["predicted_C"], linestyle="--", label="Linear fit")

ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Temperature (°C)")

ax.legend();

## Sampling Uncertainty

The fitted slope is not a property we know exactly. It is an estimate based on a finite sample containing natural variability and measurement noise.

One way to visualize this uncertainty is to repeatedly create possible new samples from the observations and refit the model. This is called **bootstrapping**.

### One bootstrap sample

A bootstrap sample is created by sampling rows with replacement. It has the same number of rows as the original dataset, but some observations may appear more than once while others may not appear at all.

In [ ]:
bootstrap_sample = df.sample(n=len(df), replace=True, random_state=5)

boot_slope, boot_intercept = np.polyfit(bootstrap_sample["elevation_m"], bootstrap_sample["temperature_C"], 1)

print(f"Original slope:  {slope:.5f}")
print(f"Bootstrap slope: {boot_slope:.5f}")

### Repeat the bootstrap many times

In [ ]:
rng_boot = np.random.default_rng(20)
bootstrap_slopes = []

for i in range(1000):

    # Generate an array of len(df) random integers between 0 and len(df)
    indices = rng_boot.integers(0, len(df), len(df)) 

    # Get the sample from the dataframe. note: some rows may not appear and some may appear more than once
    sample = df.iloc[indices]

    # Fit the sample to a linear model
    b1, b0 = np.polyfit(sample["elevation_m"], sample["temperature_C"], 1)

    # Add the slope to a list 
    bootstrap_slopes.append(b1)

bootstrap_slopes = np.array(bootstrap_slopes)
bootstrap_slopes[:5]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(bootstrap_slopes, bins=30, edgecolor="white")
ax.axvline(slope, linestyle="--", color='red', label=f"Slope from original sample")

ax.set_xlabel("Regression slope (°C per meter)")
ax.set_ylabel("Count")
ax.set_title("Distribution of the Regression Slopes")

ax.legend();

## Bootstrap Confidence Interval

A simple 95% bootstrap confidence interval can be formed from the 2.5 and 97.5 percentiles of the bootstrap distribution.

If we repeatedly collected comparable samples and constructed intervals this way, approximately 95% of those intervals would contain the true slope under the assumptions of the method.

### Calculate confidence interval from percentiles using [np.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)

In [ ]:
bootstrap_ci = np.percentile(bootstrap_slopes, [2.5, 97.5])

print(f"Slope estimate: {slope:.5f} °C per meter")
print(f"95% bootstrap CI: {bootstrap_ci[0]:.5f} to {bootstrap_ci[1]:.5f}")

## Analytical Uncertainty with [scipy.stats.linregress](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)

For simple linear regression, we can also calculate the slope, its standard error, and a p-value. These calculations rely on the linear model assumptions, including independent errors with constant variance.

In [ ]:
result = stats.linregress(df["elevation_m"], df["temperature_C"])

print(f"Slope: {result.slope:.5f}")
print(f"Slope standard error: {result.stderr:.6f}")
print(f"p-value: {result.pvalue:.3e}")

### Convert the standard error to a 95% confidence interval

For a slope estimate, a two-sided 95% interval is approximately

$$
b_1 \pm t_c * stderr(b_1)
$$

where $t_c$ is a critical value from the t-distribution with $n-2$ degrees of freedom.

In [ ]:
n = len(df)
degrees_of_freedom = n - 2

# get critical value from the students to distribution using scipy.stats
t_critical = stats.t.ppf(0.975, degrees_of_freedom)

analytical_ci = (
    result.slope - t_critical * result.stderr,
    result.slope + t_critical * result.stderr
)

print(f"95% analytical CI: {analytical_ci[0]:.5f} to {analytical_ci[1]:.5f}")
print(f"95% bootstrap CI:  {bootstrap_ci[0]:.5f} to {bootstrap_ci[1]:.5f}")

## Confidence Interval vs Prediction Interval with [statsmodels.api.OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)

Different intervals:

- a **confidence interval for the mean response** describes uncertainty in the estimated mean temperature at a given elevation
- a **prediction interval** describes where a new individual temperature observation might fall

The prediction interval is wider because it includes both uncertainty in the fitted line and the natural scatter of individual observations.

In [ ]:
X = sm.add_constant(df["elevation_m"])
model = sm.OLS(df["temperature_C"], X).fit()

x_grid = np.linspace(df["elevation_m"].min(), df["elevation_m"].max(), 10)
X_grid = sm.add_constant(x_grid)
prediction = model.get_prediction(X_grid).summary_frame(alpha=0.05)

prediction

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.fill_between(x_grid, prediction["mean_ci_lower"], prediction["mean_ci_upper"], color='orange', alpha=0.5, label="95% confidence interval")
ax.fill_between(x_grid, prediction["obs_ci_lower"], prediction["obs_ci_upper"], color='lightblue', alpha=0.25, label="95% prediction interval")

ax.scatter(df["elevation_m"], df["temperature_C"], label="Observed", alpha=0.8)
ax.plot(x_grid, prediction["mean"], label="Mean fit", linestyle="--")

ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Confidence and Prediction Intervals")

ax.legend();

## Hypothesis Testing for the Slope

A common null hypothesis for simple linear regression is

$$
H_0: b_1 = 0
$$

The alternative hypothesis is

$$
H_A: b_1 \ne 0.
$$

Test question: if the true slope were zero, how unusual would a slope this far from zero be under the model assumptions?

### What Hypothesis Is `linregress` Testing?

When we run

```python
result = stats.linregress(df["elevation_m"], df["temperature_C"])
```

we do not explicitly provide a null hypothesis. That is because `stats.linregress` has a particular hypothesis test built into the function. For the default test, the hypotheses are

$$
H_0:b_1 = 0
$$
$$
H_A:b_1 \ne 0,
$$

The test asks: Is the estimated relationship between elevation and temperature sufficiently different from a slope of zero that it would be difficult to explain by sampling variability alone?

The p-value returned as `result.pvalue` is the probability, assuming that the true slope really is zero and the regression assumptions hold, of obtaining a slope estimate at least this far from zero simply because of sampling variability.

The significance level, usually written as $\alpha$, is the threshold you choose for deciding whether a result is statistically significant.

A common choice is $\alpha = 0.05$

which means you are willing to accept a 5% probability of rejecting the null hypothesis when it is actually true. The decision rule is:

$$ p < \alpha \quad \Rightarrow \quad \text{reject } H_0 $$

If $p=0.02$ and $\alpha = 0.05$, the result is considered statistically significant. 

Note: the value of $\alpha$ should be chosen before examining the data.

In [ ]:
alpha = 0.05

print(f"p-value = {result.pvalue:.3e}")

if result.pvalue < alpha:
    print(f"The observed slope is inconsistent with H0 at the {alpha} significance level.")
else:
    print(f"The data do not provide enough evidence to reject H0 at the {alpha} level.")

### What a p-value does not tell us

- the probability that the null hypothesis is true
- a measure of whether the relationship is scientifically important

A small p-value can occur for a very small effect if the sample size is large enough. Always examine estimated effect size, uncertainty, residuals, and the scientific context.

A small p-value says: "If the true slope were zero, these data would be unusual" 
$$
P(data | H_0)
$$

It does not say: "Therefore, there is only a p percent chance the true slope is zero"
$$
P(H_0 | data)
$$

## Data with No Systematic Relationship

For comparison, create a dataset where temperature randomly varies with elevation

In [ ]:
rng_null = np.random.default_rng(33)
null_temperature = 15 + rng_null.normal(0, 2.0, len(elevation))
null_result = stats.linregress(elevation, null_temperature)

print(f"Estimated slope: {null_result.slope:.5f}")
print(f"p-value: {null_result.pvalue:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(elevation, null_temperature, label="Observed")
ax.plot(elevation, null_result.intercept + null_result.slope * elevation, linestyle="--", label="Linear fit")

ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature vs. Elevation")

ax.legend();

## p-Hacking

In class working with neighbors:

1. Select a significance level, alpha
2. x-data: create a Numpy array of random numbers of size (n_obs, n_pred), where n_obs is the number of observations and n_pred is the number of predictors
3. y-data: create a Numpy array of random numbers of size (n_obs)
5. For each predictor, find the p-value and determine if it is significant based on your alpha
6. Plot the linear regression of the smallest p-value

In [ ]:
# Work here for p-Hacking 
rng = np.random.default_rng(314)

n_obs = 100
n_pred = 100



## Multiple Ways to Fit a Linear Regression Model

We have used several Python functions that can fit a straight line to data. For simple linear regression, they will generally produce the same slope and intercept, but they are designed for different purposes.

* **`np.polyfit(x, y, 1)`** is a convenient way to fit a line when you want the fit coefficients. It can also fit higher-order polynomials by changing the polynomial degree. It is useful for quickly fitting and plotting relationships. It does not automatically provide most of the statistical information associated with regression.

* **`stats.linregress(x, y)`** is designed for simple linear regression with one predictor variable. In addition to the slope and intercept, it returns quantities such as the correlation coefficient, standard errors, and a p-value for testing whether the slope is zero. It is convenient when you want both the fit line and basic stats.

* **`sm.OLS(y, X)`** provides a more complete statistical modeling framework. It can handle multiple predictor variables and provides detailed model statistics including confidence intervals. Unlike `polyfit` and `linregress`, a column representing the intercept usually needs to be added explicitly with `sm.add_constant()`.

Use `np.polyfit` for straightforward curve fitting, `stats.linregress` for simple linear regression with basic statistical inference, and `sm.OLS` when you need a more complete analysis or multiple predictors (i.e., multiple linear regression).

## Summary

The regression line from the previous lesson is only one estimate of an underlying relationship.

Important ideas:

- coefficient from fits have sampling uncertainty
- bootstrap resampling provides a computational way to estimate that uncertainty
- confidence intervals describe uncertainty in model parameters or mean responses
- prediction intervals describe uncertainty for future individual observations
- hypothesis tests compare the observed result with what would be expected under a specified null hypothesis
- statistical significance should be interpreted together with effect size, uncertainty, residual behavior, and scientific meaning
- watch out for intentional or unintentional p-hacking

## Things to Try

1. Change the noise standard deviation from `1.5` to `3.0`. How do the confidence interval and p-value change?
2. Reduce the sample from 50 observations to 15. What happens to slope uncertainty?
3. Repeat the bootstrap with 200, 2,000, and 10,000 resamples. Which quantities change noticeably?
4. what happens as you change the number of observations and predictors in the p-hacking demonstration